# NL-SQL Development & Testing Notebook

**Branch:** devil  
**Purpose:** Fast prototyping and testing of NL-SQL query generation

This notebook provides a quick way to:
- Test natural language queries
- Inspect generated SQL
- View query results
- Debug issues
- Test new features

---

## 1. Setup & Imports

In [ ]:
import requests
import json
import pandas as pd
from IPython.display import display, HTML, JSON
import time

# API Configuration
API_BASE = "http://localhost:8001"

# Pretty print helper
def pretty_print(data):
    """Pretty print JSON data"""
    print(json.dumps(data, indent=2, default=str))

# Test API connection
try:
    response = requests.get(f"{API_BASE}/health")
    print("✅ API Connection: SUCCESS")
    print(f"Status: {response.status_code}")
except Exception as e:
    print(f"❌ API Connection: FAILED")
    print(f"Error: {e}")
    print("\n💡 Make sure containers are running: docker-compose up -d")

## 2. Quick Query Tester

Test any natural language query quickly.

In [ ]:
def test_query(question, show_sql=True, show_results=True, show_viz=False, model="gpt-4o"):
    """
    Test a natural language query
    
    Args:
        question: Natural language query
        show_sql: Show generated SQL
        show_results: Show query results as DataFrame
        show_viz: Show visualization recommendation
        model: LLM model to use
    """
    print("="*80)
    print(f"📝 QUESTION: {question}")
    print("="*80)
    
    start_time = time.time()
    
    try:
        # Call API
        response = requests.post(
            f"{API_BASE}/api/query",
            json={
                "question": question,
                "include_explanation": True,
                "model": model,
                "skip_spell_check": True
            },
            timeout=60
        )
        
        elapsed = (time.time() - start_time) * 1000
        
        result = response.json()
        
        # Check for errors
        if not result.get('success', False):
            print("\n❌ QUERY FAILED")
            print(f"Error: {result.get('error', 'Unknown error')}")
            
            # Check for ambiguity
            if result.get('needs_clarification'):
                print("\n💡 CLARIFICATION NEEDED:")
                ambiguity = result.get('ambiguity', {})
                print(f"Reason: {ambiguity.get('reason', 'N/A')}")
                print("\nSuggested refinements:")
                for idx, opt in enumerate(ambiguity.get('clarification_options', []), 1):
                    print(f"  {idx}. {opt.get('text', 'N/A')}")
                    print(f"     Query: {opt.get('refined_query', 'N/A')}")
            return None
        
        # Show SQL
        if show_sql:
            print("\n🔍 GENERATED SQL:")
            print("-" * 80)
            print(result.get('sql', 'N/A'))
            print("-" * 80)
        
        # Show explanation
        if result.get('explanation'):
            print(f"\n💬 EXPLANATION: {result['explanation']}")
        
        # Show metadata
        print(f"\n📊 METADATA:")
        print(f"  • Model: {result.get('model', 'N/A')}")
        print(f"  • Tokens: {result.get('tokens_used', 'N/A')}")
        print(f"  • Execution: {result.get('execution_time_ms', 0):.2f}ms")
        print(f"  • Total: {elapsed:.2f}ms")
        print(f"  • Rows: {result.get('row_count', 0)}")
        
        # Show visualization
        if show_viz and result.get('visualization'):
            print("\n📈 VISUALIZATION:")
            viz = result['visualization']
            print(f"  • Chart Type: {viz.get('recommended_chart', 'N/A')}")
            print(f"  • Reason: {viz.get('reason', 'N/A')}")
            if viz.get('number_format'):
                nf = viz['number_format']
                print(f"  • Currency: {nf.get('isCurrency', False)}")
                print(f"  • K/M Notation: {nf.get('useShortNumbers', False)}")
                print(f"  • Decimals: {nf.get('decimalPlaces', 0)}")
        
        # Show results as DataFrame
        if show_results and result.get('rows'):
            print("\n📋 RESULTS:")
            df = pd.DataFrame(result['rows'])
            print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            display(df.head(20))  # Show first 20 rows
        
        print("\n" + "="*80)
        print("✅ SUCCESS")
        print("="*80)
        
        return result
        
    except requests.exceptions.Timeout:
        print("\n⏱️  TIMEOUT: Query took too long (>60s)")
        return None
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Example usage
print("Function 'test_query()' loaded!")
print("\nExample usage:")
print('  test_query("How many patients do we have?")')
print('  test_query("Show me patients in Florida", show_results=True)')

## 3. Test Cases Library

Pre-defined test cases for common queries.

In [ ]:
# Test case categories
TEST_CASES = {
    "basic_counts": [
        "How many patients do we have?",
        "Count of patients with diabetes",
        "How many ER visits in 2025?"
    ],
    "demographics": [
        "Show me patients by state",
        "Patients by gender and age bucket",
        "Show me patients in California"
    ],
    "costs": [
        "Total cost by month",
        "Average cost per patient",
        "Top 10 most expensive patients"
    ],
    "conditions": [
        "Show me diabetic patients",
        "Cancer prevalence by type",
        "Patients with hypertension over 65"
    ],
    "utilization": [
        "ER frequent flyers",
        "Patients with 3 or more ER visits",
        "Top 20 patients by ER visits"
    ],
    "quality": [
        "Osteoporosis screening rate",
        "Percentage of women 65-75 who received screening"
    ],
    "geographic": [
        "Top 100 patients per county",
        "Patients by county and state"
    ]
}

def list_test_cases():
    """List all available test cases"""
    print("📚 TEST CASE LIBRARY\n")
    for category, queries in TEST_CASES.items():
        print(f"\n{category.upper().replace('_', ' ')}:")
        for idx, query in enumerate(queries, 1):
            print(f"  {idx}. {query}")

def run_test_suite(category=None, max_queries=None):
    """Run a test suite"""
    if category:
        if category not in TEST_CASES:
            print(f"❌ Unknown category: {category}")
            print(f"Available: {list(TEST_CASES.keys())}")
            return
        queries = TEST_CASES[category]
        print(f"🧪 Running test suite: {category.upper()}")
    else:
        queries = [q for queries in TEST_CASES.values() for q in queries]
        print(f"🧪 Running FULL test suite")
    
    if max_queries:
        queries = queries[:max_queries]
    
    print(f"Total queries: {len(queries)}\n")
    
    results = []
    for idx, query in enumerate(queries, 1):
        print(f"\n[{idx}/{len(queries)}] Testing: {query}")
        result = test_query(query, show_sql=False, show_results=False, show_viz=False)
        results.append({
            'query': query,
            'success': result is not None and result.get('success', False),
            'rows': result.get('row_count', 0) if result else 0,
            'time_ms': result.get('execution_time_ms', 0) if result else 0
        })
        time.sleep(1)  # Rate limiting
    
    # Summary
    df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("📊 TEST SUITE SUMMARY")
    print("="*80)
    print(f"Total: {len(results)}")
    print(f"Success: {df['success'].sum()} ({df['success'].mean()*100:.1f}%)")
    print(f"Failed: {(~df['success']).sum()}")
    print(f"Avg Time: {df['time_ms'].mean():.2f}ms")
    display(df)

list_test_cases()

## 4. Quick Tests

Run your test queries here!

In [ ]:
# Test 1: Simple count
test_query("How many patients do we have?")

In [ ]:
# Test 2: Demographics
test_query("Show me patients by state", show_results=True)

In [ ]:
# Test 3: Cost analysis
test_query("Total cost by month", show_results=True, show_viz=True)

In [ ]:
# Test 4: Top patients
test_query("Top 10 most expensive patients", show_results=True)

In [ ]:
# Test 5: Conditions
test_query("Show me diabetic patients over 65", show_results=True)

## 5. Direct SQL Execution

Test SQL queries directly without NL conversion.

In [ ]:
def execute_sql(sql, show_results=True):
    """
    Execute SQL directly
    
    Args:
        sql: SQL query to execute
        show_results: Show results as DataFrame
    """
    print("="*80)
    print("🔧 DIRECT SQL EXECUTION")
    print("="*80)
    print(sql)
    print("-" * 80)
    
    start_time = time.time()
    
    try:
        response = requests.post(
            f"{API_BASE}/api/sql",
            json={"sql": sql, "max_results": 100},
            timeout=30
        )
        
        elapsed = (time.time() - start_time) * 1000
        
        result = response.json()
        
        if not result.get('success', False):
            print(f"\n❌ FAILED: {result.get('error', 'Unknown error')}")
            return None
        
        print(f"\n✅ SUCCESS")
        print(f"  • Rows: {result.get('row_count', 0)}")
        print(f"  • Time: {elapsed:.2f}ms")
        
        if show_results and result.get('rows'):
            df = pd.DataFrame(result['rows'])
            print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
            display(df)
        
        return result
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        return None

# Example
# execute_sql("SELECT COUNT(*) as patient_count FROM vw_patients_2025")

## 6. Schema Explorer

Explore available views and columns.

In [ ]:
def explore_schema():
    """Get all view schemas"""
    sql = """
        SELECT 
            table_name,
            column_name,
            data_type
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name LIKE 'vw_%'
        ORDER BY table_name, ordinal_position;
    """
    
    result = execute_sql(sql, show_results=False)
    
    if result and result.get('rows'):
        df = pd.DataFrame(result['rows'])
        
        print("\n📊 AVAILABLE VIEWS:\n")
        for view_name in df['table_name'].unique():
            view_cols = df[df['table_name'] == view_name]
            print(f"\n{view_name}:")
            for _, row in view_cols.iterrows():
                print(f"  • {row['column_name']} ({row['data_type']})")
        
        return df
    return None

# Uncomment to explore schema
# explore_schema()

## 7. Semantic Dictionary Viewer

View semantic terms and mappings.

In [ ]:
def view_semantic_dictionary():
    """View semantic dictionary"""
    sql = """
        SELECT 
            term,
            term_type,
            view_name,
            column_name,
            example_usage
        FROM nlsql_semantic_alias
        WHERE is_active = TRUE
        ORDER BY term_type, term
        LIMIT 100;
    """
    
    result = execute_sql(sql, show_results=False)
    
    if result and result.get('rows'):
        df = pd.DataFrame(result['rows'])
        print(f"\n📚 SEMANTIC DICTIONARY ({len(df)} terms)\n")
        display(df)
        return df
    return None

# Uncomment to view dictionary
# view_semantic_dictionary()

## 8. Query Templates Viewer

View query templates.

In [ ]:
def view_query_templates():
    """View query templates"""
    sql = """
        SELECT 
            natural_language_pattern,
            category,
            example_input,
            description
        FROM nlsql_query_templates
        WHERE is_active = TRUE
        ORDER BY category;
    """
    
    result = execute_sql(sql, show_results=False)
    
    if result and result.get('rows'):
        df = pd.DataFrame(result['rows'])
        print(f"\n📋 QUERY TEMPLATES ({len(df)} templates)\n")
        display(df)
        return df
    return None

# Uncomment to view templates
# view_query_templates()

## 9. Performance Benchmarking

Benchmark query performance.

In [ ]:
def benchmark_queries(queries, iterations=3):
    """
    Benchmark multiple queries
    
    Args:
        queries: List of queries to test
        iterations: Number of times to run each query
    """
    print(f"🏃 BENCHMARKING {len(queries)} queries × {iterations} iterations\n")
    
    results = []
    
    for query in queries:
        print(f"Testing: {query}")
        times = []
        
        for i in range(iterations):
            start = time.time()
            result = test_query(query, show_sql=False, show_results=False, show_viz=False)
            elapsed = (time.time() - start) * 1000
            times.append(elapsed)
            time.sleep(0.5)  # Rate limiting
        
        results.append({
            'query': query,
            'avg_ms': sum(times) / len(times),
            'min_ms': min(times),
            'max_ms': max(times)
        })
    
    df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("📊 BENCHMARK RESULTS")
    print("="*80)
    display(df.sort_values('avg_ms'))
    return df

# Example
# benchmark_queries([
#     "How many patients do we have?",
#     "Total cost by month",
#     "Top 10 most expensive patients"
# ], iterations=3)

## 10. Custom Test Area

Use this cell for your custom tests and experiments.

In [ ]:
# Your custom tests here



---

## 📚 Quick Reference

### Main Functions:

```python
# Test a single query
test_query("How many patients?")

# Test with options
test_query("Show patients", show_sql=True, show_results=True, show_viz=True)

# List test cases
list_test_cases()

# Run test suite
run_test_suite(category="basic_counts")
run_test_suite()  # Run all

# Execute SQL directly
execute_sql("SELECT COUNT(*) FROM vw_patients_2025")

# Explore schema
explore_schema()

# View semantic dictionary
view_semantic_dictionary()

# View query templates
view_query_templates()

# Benchmark queries
benchmark_queries(["query1", "query2"], iterations=3)
```

### Test Categories:
- `basic_counts` - Simple count queries
- `demographics` - Patient demographics
- `costs` - Cost analysis
- `conditions` - Medical conditions
- `utilization` - Healthcare utilization
- `quality` - Quality measures
- `geographic` - Geographic analysis

---